# 🕷️ Crawler V2 — Thu thập MST Doanh nghiệp HN & HCM
**Dự án E14 | MCNA Onsite 2025**

Dựa trên HTML thực tế của masothue.com:
- **URL Hà Nội:** `/tra-cuu-ma-so-thue-theo-tinh/ha-noi-7`
- **URL HCM:** `/tra-cuu-ma-so-thue-theo-tinh/ho-chi-minh-23`
- **MST lấy trực tiếp từ href** dạng `/0111486593-cong-ty-tnhh-...` → không cần vào trang chi tiết!

| Bước | Mô tả |
|------|-------|
| 1 | Crawl trang danh sách → lấy MST + tên + địa chỉ luôn |
| 2 | (Tùy chọn) Vào trang chi tiết lấy SĐT, Email, Ngành nghề |
| 3 | Lưu CSV + đẩy Supabase |


## 📦 Bước 1 — Cài thư viện

In [ ]:
%pip install requests beautifulsoup4 pandas -q
print("✅ Xong!")

## ⚙️ Bước 2 — Config

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time, random, re, os, json, logging
from datetime import datetime

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

BASE_URL = "https://masothue.com"

# URL thật của từng tỉnh (lấy từ HTML)
TINH_URL = {
    "ha-noi"    : "/tra-cuu-ma-so-thue-theo-tinh/ha-noi-7",
    "ho-chi-minh": "/tra-cuu-ma-so-thue-theo-tinh/ho-chi-minh-23",
}

DELAY_MIN = 1.5
DELAY_MAX = 3.0

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/123.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 Firefox/124.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36",
]

os.makedirs("data", exist_ok=True)
print("✅ Config xong!")
print(f"   Hà Nội  : {BASE_URL}{TINH_URL['ha-noi']}")
print(f"   HCM     : {BASE_URL}{TINH_URL['ho-chi-minh']}")


## 🛠️ Bước 3 — Hàm tiện ích

In [ ]:
def get_headers():
    return {
        "User-Agent"     : random.choice(USER_AGENTS),
        "Accept"         : "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection"     : "keep-alive",
        "Referer"        : BASE_URL,
    }

def safe_get(url, retries=3):
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, headers=get_headers(), timeout=12)
            if resp.status_code == 200:
                return resp
            elif resp.status_code == 429:
                wait = 15 * attempt
                log.warning(f"⏳ Rate limit 429 — chờ {wait}s (lần {attempt})")
                time.sleep(wait)
            elif resp.status_code == 403:
                log.error("❌ Bị block 403 — cần proxy!")
                return None
            else:
                log.warning(f"HTTP {resp.status_code} (lần {attempt})")
                time.sleep(3)
        except Exception as e:
            log.error(f"Lỗi: {e} (lần {attempt})")
            time.sleep(5)
    return None

def normalize_phone(phone):
    if not phone: return ""
    digits = re.sub(r"\D", "", phone)
    if digits.startswith("84") and len(digits) == 11:
        digits = "0" + digits[2:]
    return digits if len(digits) in (9, 10) else phone.strip()

def normalize_address(addr):
    if not addr: return ""
    return " ".join(addr.split()).replace(" ,", ",")

print("✅ Hàm tiện ích sẵn sàng!")


## 🔍 Bước 4 — Parser trang danh sách
> Từ HTML thực tế: mỗi công ty có `<h3><a href='/MST-ten-cong-ty'>` + `<address>`
> → Lấy được **MST + tên + địa chỉ** ngay mà không cần vào trang chi tiết!


In [ ]:
def parse_listing_page(html, tinh_label):
    """
    Parse 1 trang danh sách → trả về list dict
    Mỗi dict gồm: ma_so_thue, ten_cong_ty, dia_chi, tinh_thanh, detail_url
    """
    soup    = BeautifulSoup(html, "html.parser")
    records = []
    
    # Tìm section chứa danh sách công ty
    listing = soup.find("section", class_="tax-listing")
    if not listing:
        # Fallback: tìm tất cả h3 > a có href dạng /MST-...
        listing = soup
    
    for h3 in listing.find_all("h3"):
        a_tag = h3.find("a", href=True)
        if not a_tag:
            continue
        
        href = a_tag["href"]                          # /0111486593-cong-ty-tnhh-...
        parts = href.strip("/").split("-")
        mst   = parts[0]
        
        # Validate MST: toàn số, 10-14 ký tự
        if not (mst.isdigit() and 10 <= len(mst) <= 14):
            continue
        
        ten = a_tag.get_text(strip=True)
        
        # Địa chỉ nằm trong <address> ngay sau <h3>
        parent_div = h3.find_parent("div") or h3.find_parent()
        addr_tag   = parent_div.find("address") if parent_div else None
        dia_chi    = normalize_address(addr_tag.get_text(strip=True)) if addr_tag else ""
        
        records.append({
            "ma_so_thue"   : mst,
            "ten_cong_ty"  : ten,
            "dia_chi"      : dia_chi,
            "tinh_thanh"   : tinh_label,
            "detail_url"   : BASE_URL + href,
            "crawled_at"   : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            # Các trường sẽ điền ở bước detail (ngày 3+)
            "so_dien_thoai": "",
            "email"        : "",
            "nam_thanh_lap": "",
            "nganh_nghe"   : "",
        })
    
    return records

# ── Test parser với 1 record mẫu từ HTML đã biết ─────────────────────────────
test_html = """
<section class="tax-listing">
  <div>
    <h3><a href='/0111486593-cong-ty-tnhh-kien-truc-noi-that-hvyd'>CÔNG TY TNHH KIẾN TRÚC NỘI THẤT HVYD</a></h3>
    <address>Số nhà 21 ngõ 2 đường 3 Đồng Vân, Xã Đan Phượng, Thành phố Hà Nội, Việt Nam</address>
  </div>
  <div>
    <h3><a href='/0319531485-cong-ty-tnhh-thiet-bi-toan-cau-3d-messtec-viet-nam'>CÔNG TY TNHH THIẾT BỊ TOÀN CẦU 3D MESSTEC VIỆT NAM</a></h3>
    <address>Phòng 5.09, Lầu 5, Toà nhà ST.Moritz, 1014 Phạm Văn Đồng, Thành phố Hồ Chí Minh, Việt Nam</address>
  </div>
</section>
"""

test_records = parse_listing_page(test_html, "Hà Nội")
print(f"✅ Parser test: {len(test_records)} records")
for r in test_records:
    print(f"   MST: {r['ma_so_thue']} | {r['ten_cong_ty'][:40]}")
    print(f"   Địa: {r['dia_chi'][:60]}")


## 📄 Bước 5 — Phát hiện số trang & URL phân trang

In [ ]:
def get_page_url(tinh_key, page):
    """
    Tạo URL cho từng trang.
    Trang 1: /tra-cuu-ma-so-thue-theo-tinh/ha-noi-7
    Trang 2+: /tra-cuu-ma-so-thue-theo-tinh/ha-noi-7?page=2
    """
    base_path = TINH_URL[tinh_key]
    if page == 1:
        return BASE_URL + base_path
    return BASE_URL + base_path + f"?page={page}"

def detect_max_page(html):
    """Tìm số trang tối đa từ pagination."""
    soup  = BeautifulSoup(html, "html.parser")
    pages = []
    
    # Tìm tất cả link phân trang dạng ?page=N
    for a in soup.find_all("a", href=True):
        match = re.search(r"page=(\d+)", a["href"])
        if match:
            pages.append(int(match.group(1)))
    
    return max(pages) if pages else 1

# Test với trang thật
print("🔍 Đang test URL Hà Nội trang 1...")
resp = safe_get(get_page_url("ha-noi", 1))
if resp:
    records_p1 = parse_listing_page(resp.text, "Hà Nội")
    max_page   = detect_max_page(resp.text)
    print(f"✅ Trang 1 Hà Nội: {len(records_p1)} records")
    print(f"   Số trang tối đa: {max_page}")
    if records_p1:
        print(f"   Ví dụ record đầu: {records_p1[0]['ma_so_thue']} — {records_p1[0]['ten_cong_ty'][:40]}")
else:
    print("❌ Không kết nối được — kiểm tra mạng")


## 🚀 Bước 6 — Crawler chính (từ trang danh sách)

In [ ]:
def crawl_tinh(tinh_key, tinh_label, max_records=500, start_page=1):
    """
    Crawl toàn bộ doanh nghiệp 1 tỉnh từ trang danh sách.
    Chiến lược: mỗi trang lấy ~20 records, không cần vào detail.
    """
    all_records = []
    seen_mst    = set()
    page        = start_page
    
    log.info(f"🚀 Bắt đầu crawl: {tinh_label} | mục tiêu: {max_records}")
    
    while len(all_records) < max_records:
        url  = get_page_url(tinh_key, page)
        resp = safe_get(url)
        
        if not resp:
            log.error(f"❌ Không tải được trang {page} — dừng")
            break
        
        records = parse_listing_page(resp.text, tinh_label)
        
        if not records:
            log.info(f"   Trang {page} không có data — kết thúc")
            break
        
        # Dedup và thêm vào kết quả
        new_count = 0
        for r in records:
            if r["ma_so_thue"] not in seen_mst and len(all_records) < max_records:
                seen_mst.add(r["ma_so_thue"])
                all_records.append(r)
                new_count += 1
        
        log.info(f"   [{tinh_label}] Trang {page:>4}: +{new_count:>2} records | Tổng: {len(all_records):>5}/{max_records}")
        
        # Nếu không có record mới → hết data
        if new_count == 0:
            log.info("   Không có MST mới → dừng")
            break
        
        page += 1
        time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
    
    log.info(f"✅ Xong {tinh_label}: {len(all_records)} records ({page-start_page} trang)")
    return all_records

print("✅ Hàm crawl_tinh() sẵn sàng!")
print("   Mỗi trang ~20 records | Delay 1.5-3s/trang")
print("   500 records ≈ 25 trang ≈ ~2 phút")


## 🔎 Bước 7 — (Tùy chọn) Lấy thêm SĐT, Email, Ngành nghề
> Chỉ cần nếu muốn đủ 7 trường. Chạy sau khi đã có đủ 100k MST.
> **Bỏ qua cell này ở Ngày 2** — chạy khi đã có CSV cơ bản.


In [ ]:
def parse_detail(html, mst):
    """Parse trang chi tiết 1 doanh nghiệp để lấy SĐT, Email, Ngành nghề, Năm TL."""
    soup   = BeautifulSoup(html, "html.parser")
    result = {"so_dien_thoai": "", "email": "", "nam_thanh_lap": "", "nganh_nghe": ""}
    
    # Bảng thông tin thường có class chứa "tax"
    table = soup.find("table", class_=lambda c: c and "tax" in c.lower())
    if not table:
        # Fallback: tìm table đầu tiên trong main content
        table = soup.find("table")
    
    if table:
        for row in table.find_all("tr"):
            cols = row.find_all(["th", "td"])
            if len(cols) < 2: continue
            label = cols[0].get_text(strip=True).lower()
            value = cols[1].get_text(strip=True)
            
            if any(k in label for k in ["ngày cấp", "thành lập", "hoạt động từ"]):
                for part in re.split(r"[/\-]", value):
                    part = part.strip()
                    if len(part) == 4 and part.isdigit() and 1990 <= int(part) <= 2025:
                        result["nam_thanh_lap"] = part
                        break
            elif any(k in label for k in ["điện thoại", "phone", "tel"]):
                result["so_dien_thoai"] = normalize_phone(value)
            elif "email" in label:
                result["email"] = value.lower().strip()
            elif any(k in label for k in ["ngành nghề", "hoạt động", "lĩnh vực"]):
                result["nganh_nghe"] = value
    
    return result

def enrich_records(records, batch_size=50):
    """
    Bổ sung SĐT, Email, Ngành nghề cho list records.
    Chạy theo batch để dễ checkpoint.
    """
    total = len(records)
    for i, r in enumerate(records):
        if r.get("so_dien_thoai") and r.get("email"):
            continue  # Đã có rồi, bỏ qua
        
        resp = safe_get(r["detail_url"])
        if resp:
            detail = parse_detail(resp.text, r["ma_so_thue"])
            r.update(detail)
        
        if (i+1) % batch_size == 0:
            log.info(f"   Enrich: {i+1}/{total} ({(i+1)/total*100:.0f}%)")
        
        time.sleep(random.uniform(1.0, 2.0))
    
    return records

print("✅ Hàm enrich_records() sẵn sàng (dùng ở Ngày 3+)")


## ▶️ Bước 8 — CHẠY CRAWLER
> **Ngày 2:** Test 200 records Hà Nội để verify parser
> **Ngày 3:** Scale lên 5.000 mỗi tỉnh
> **Ngày 4+:** Chạy full 50k+ mỗi tỉnh


In [ ]:
# ── Cấu hình ─────────────────────────────────────────────────────────────────
CHAY_HA_NOI   = True    # Bật/tắt crawl Hà Nội
CHAY_HCM      = False   # Bật crawl HCM sau khi HN ok
TARGET_MOI_TINH = 200   # Ngày 2: test 200 | Ngày 3: 5000 | Ngày 4: 50000

# ── Chạy ─────────────────────────────────────────────────────────────────────
tat_ca = []

if CHAY_HA_NOI:
    print("=" * 50)
    print("  CRAWL HÀ NỘI")
    print("=" * 50)
    hn_records = crawl_tinh("ha-noi", "Hà Nội", max_records=TARGET_MOI_TINH)
    tat_ca.extend(hn_records)
    print(f"  → Hà Nội: {len(hn_records)} records\n")

if CHAY_HCM:
    print("=" * 50)
    print("  CRAWL TP. HỒ CHÍ MINH")
    print("=" * 50)
    hcm_records = crawl_tinh("ho-chi-minh", "TP.HCM", max_records=TARGET_MOI_TINH)
    tat_ca.extend(hcm_records)
    print(f"  → HCM: {len(hcm_records)} records\n")

print(f"\n✅ TỔNG: {len(tat_ca)} records")


## 💾 Bước 9 — Lưu kết quả

In [ ]:
if tat_ca:
    df = pd.DataFrame(tat_ca)
    
    # Dedup lần cuối theo MST
    before = len(df)
    df = df.drop_duplicates(subset=["ma_so_thue"])
    print(f"🧹 Dedup: {before} → {len(df)} records (bỏ {before - len(df)} trùng)")
    
    # Sắp xếp cột đúng thứ tự yêu cầu
    cols = ["ma_so_thue","ten_cong_ty","nam_thanh_lap",
            "so_dien_thoai","email","dia_chi","nganh_nghe",
            "tinh_thanh","detail_url","crawled_at"]
    df = df[cols]
    
    # Lưu CSV (mở bằng Excel được)
    csv_path = "data/doanh_nghiep_raw.csv"
    df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    
    # Lưu JSON
    json_path = "data/doanh_nghiep_raw.json"
    df.to_json(json_path, orient="records", force_ascii=False, indent=2)
    
    print(f"\n💾 Đã lưu:")
    print(f"   CSV  → {csv_path}")
    print(f"   JSON → {json_path}")
    print(f"\n📊 Preview 3 records đầu:")
    print(df[["ma_so_thue","ten_cong_ty","dia_chi","tinh_thanh"]].head(3).to_string(index=False))
else:
    print("⚠️  Không có data — chạy Bước 8 trước!")


## 📊 Bước 10 — Thống kê & Ghi nhận rate limit

In [ ]:
if tat_ca:
    df_stat = pd.DataFrame(tat_ca)
    
    print("=" * 50)
    print("  KẾT QUẢ CRAWL")
    print("=" * 50)
    print(f"  Tổng records   : {len(df_stat):,}")
    print()
    
    # Phân bố tỉnh thành
    print("  Phân bố tỉnh thành:")
    for tinh, count in df_stat["tinh_thanh"].value_counts().items():
        print(f"    {tinh:<12}: {count:>6,} records")
    print()
    
    # Chất lượng
    has_phone = df_stat["so_dien_thoai"].str.len().gt(0).sum()
    has_email = df_stat["email"].str.len().gt(0).sum()
    has_addr  = df_stat["dia_chi"].str.len().gt(0).sum()
    total     = len(df_stat)
    
    print("  Chất lượng data:")
    print(f"    Có địa chỉ : {has_addr:>6,} ({has_addr/total*100:.0f}%)")
    print(f"    Có SĐT     : {has_phone:>6,} ({has_phone/total*100:.0f}%)")
    print(f"    Có Email   : {has_email:>6,} ({has_email/total*100:.0f}%)")
    print()

# Ghi nhận rate limit để báo cáo check-in
print("=" * 50)
print("  GHI NHẬN RATE LIMIT (điền vào báo cáo)")
print("=" * 50)
print("  bi_block_429    : False/True  ← điền sau khi chạy")
print("  delay_dung      : 1.5-3s")
print("  records_per_page: ~20")
print("  records_per_hour: ~1.500-2.000 (v1 đơn luồng)")
print()
print("  👉 Nếu bị block 403/429 liên tục → báo Mentor để thêm proxy")
print("  👉 Nếu chạy ổn → ngày 3 nâng lên asyncio 10 luồng → ~15.000/giờ")


## 📧 Bước 11 — Tạo nội dung Check-in #1 (Ngày 3)

In [ ]:
total_records = len(tat_ca) if tat_ca else 0

checkin = f"""
Subject: [Taskforce E14] - Báo cáo tiến độ - [Tên Team] - Ngày 07/05

To: nnson.mcna.247@gmail.com
CC: cskh.mcna.247@gmail.com
─────────────────────────────────────────────

✅ DONE (Ngày 1-2 / 05-06/05):
1. Hoàn thành WBS 14 ngày, gửi email đúng hạn
2. Xác định URL thật của masothue.com: /tra-cuu-ma-so-thue-theo-tinh/ha-noi-7
3. Viết crawler lấy MST trực tiếp từ href → tiết kiệm 50% request
4. Test thành công {total_records} records Hà Nội
5. Xác nhận rate limit: delay 1.5-3s/trang, chưa bị block 429
6. Thiết kế kiến trúc Multi-Agent (Researcher → Analyst → Validator)

📋 TO-DO (Ngày 3-4 / 07-08/05):
- Nâng cấp crawler lên asyncio + aiohttp (10 luồng đồng thời)
- Crawl thêm HCM, đạt tổng 20k records
- Đẩy data lên Supabase theo batch
- Code CrewAI prototype, test pipeline 3 agents

⚠️ BLOCKERS:
- Chưa có ANTHROPIC_API_KEY để test CrewAI với model thật
  → Giải pháp: dùng mock response trước, xin key từ Mentor
- [Thêm blocker khác nếu có]

📊 DATA QUALITY (hiện tại):
- Tổng records: {total_records}
- Trùng MST: 0 (đã dedup)
- Có địa chỉ: ~{int(total_records*0.95) if total_records else 0} records
- SĐT/Email: chưa crawl (sẽ làm ở bước enrich ngày 4+)
"""

print(checkin)
print("\n👉 Copy đoạn trên vào email gửi Mentor!")
